# Creating Designs by Leveraging OpenAI and Gradio UI

## Project Overview
This project demonstrates how to build a web-based platform that combines **OpenAI's DALL-E API** with **Gradio's web interface** to generate bespoke images and designs for banners and posters from text prompts.

**Use Case:** Generating customized visual content for Netflix promotional campaigns.

---

## Situation
As a designer at a cutting-edge creative agency, this platform revolutionizes visual content creation for digital marketing campaigns for high-profile clients like Netflix.

## Task
Create a platform that generates bespoke images and designs for banners and posters by entering text prompts using OpenAI's DALL-E model.

## Step 1: Install Required Libraries

Install the necessary packages if not already available.

In [ ]:
# Install required libraries
%pip install openai gradio Pillow requests

## Step 2: Import Essential Libraries

Import all necessary libraries:
- **gradio** – for building the web interface
- **requests** – for fetching image data via HTTP
- **PIL (Pillow)** – for image processing
- **io.BytesIO** – for managing binary data streams
- **openai** – for accessing the OpenAI DALL-E API

In [ ]:
import gradio as gr
import requests
from PIL import Image
from io import BytesIO
import openai
import os

print("All libraries imported successfully!")
print(f"Gradio version: {gr.__version__}")
print(f"OpenAI version: {openai.__version__}")

## Step 3: Configure OpenAI API Key

Set your OpenAI API key. It is best practice to load it from an environment variable rather than hardcoding it in the notebook.

In [ ]:
# Load API key from environment variable (recommended) or set directly
# To set the environment variable, run in your terminal:
#   export OPENAI_API_KEY="your-api-key-here"

api_key = os.environ.get("OPENAI_API_KEY")

if not api_key:
    # Fallback: prompt user to enter key (for notebook usage only)
    api_key = input("Enter your OpenAI API key: ").strip()

client = openai.OpenAI(api_key=api_key)

print("OpenAI client configured successfully!")

## Step 4: Create the `generate_image` Function

This function handles the entire image generation process:
1. Takes a text prompt as input
2. Calls OpenAI's DALL-E API
3. Fetches the generated image via its URL
4. Returns the image as a PIL Image object for display

In [ ]:
def generate_image(prompt: str) -> Image.Image:
    """
    Generate an image using OpenAI's DALL-E model based on the provided text prompt.

    Args:
        prompt (str): A text description of the image to generate.

    Returns:
        PIL.Image.Image: The generated image as a PIL Image object.
    """
    if not prompt or prompt.strip() == "":
        raise gr.Error("Please enter a text prompt to generate an image.")

    # Call the OpenAI DALL-E API to generate an image
    response = client.images.generate(
        model="dall-e-3",       # Use DALL-E 3 for high-quality images
        prompt=prompt,
        size="1024x1024",       # Image resolution
        quality="standard",     # Image quality: "standard" or "hd"
        n=1                     # Number of images to generate
    )

    # Extract the image URL from the API response
    image_url = response.data[0].url

    # Fetch the image data from the URL using the requests library
    image_response = requests.get(image_url, timeout=30)
    image_response.raise_for_status()

    # Convert the binary image data to a PIL Image using BytesIO
    image = Image.open(BytesIO(image_response.content))

    return image


print("generate_image function defined successfully!")

## Step 5: Set Up and Launch the Gradio Interface

Configure the Gradio interface with:
- **Input:** A textbox for entering the design prompt
- **Output:** An image display component for the generated design
- **Examples:** Pre-filled Netflix campaign prompt examples

In [ ]:
# Define example prompts for Netflix campaign designs
example_prompts = [
    ["A cinematic Netflix banner for a sci-fi thriller series set in space, featuring a lone astronaut silhouette against a glowing nebula, dark and dramatic mood, ultra HD"],
    ["A vibrant Netflix poster for a romantic comedy, featuring a couple in Paris under the Eiffel Tower at sunset, warm pastel colors, elegant typography space at top"],
    ["A mysterious Netflix promotional banner for a crime documentary, dark moody atmosphere, red and black color palette, shadowy figures in a rain-soaked city alley"],
    ["A bold Netflix action movie poster with an explosion in the background, a hero standing in the foreground, intense orange and black color scheme, cinematic style"],
    ["A magical Netflix fantasy series banner with enchanted forest, glowing fairy lights, mystical creatures, rich jewel tones, ethereal dreamlike atmosphere"]
]

# Create the Gradio Interface
interface = gr.Interface(
    fn=generate_image,
    inputs=gr.Textbox(
        lines=4,
        placeholder="Enter a creative text prompt to generate a Netflix campaign design...\n\nExample: A dramatic Netflix banner for an action thriller, dark atmosphere, red Netflix logo, cinematic poster style",
        label="Design Prompt"
    ),
    outputs=gr.Image(
        type="pil",
        label="Generated Design"
    ),
    title="Netflix Campaign Design Generator",
    description=(
        "### Powered by OpenAI DALL-E & Gradio\n"
        "Transform your creative ideas into stunning visual designs for Netflix campaigns. "
        "Enter a descriptive text prompt below and click **Submit** to generate a bespoke banner or poster design."
    ),
    examples=example_prompts,
    cache_examples=False,
    theme=gr.themes.Soft(),
    flagging_mode="never"
)

# Launch the Gradio interface
interface.launch(share=False)

## Step 6: Test the Image Generation (Optional)

Run a quick test of the `generate_image` function outside of the Gradio interface to verify it works correctly.

In [ ]:
import matplotlib.pyplot as plt

# Test prompt for a Netflix campaign design
test_prompt = "A dramatic Netflix original series banner featuring a lone detective in a neon-lit cyberpunk city, dark atmospheric lighting, rain-soaked streets, cinematic composition"

print(f"Generating image for prompt:\n'{test_prompt}'\n")
print("Please wait...")

# Generate the image
test_image = generate_image(test_prompt)

# Display the image using matplotlib
plt.figure(figsize=(10, 10))
plt.imshow(test_image)
plt.axis("off")
plt.title("Generated Netflix Campaign Design", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nImage generated and displayed successfully!")

## Result & Summary

This notebook demonstrates the end-to-end workflow for leveraging OpenAI's DALL-E and Gradio UI to create a design generation platform:

| Component | Role |
|-----------|------|
| **OpenAI DALL-E 3** | AI model that transforms text prompts into high-quality images |
| **Gradio** | Provides the interactive web interface for user interaction |
| **requests** | Fetches the generated image from OpenAI's CDN URL |
| **PIL (Pillow)** | Handles image loading and processing from binary data |
| **io.BytesIO** | Manages binary image streams in memory |

### Key Outcomes
- Built a fully functional web UI for AI-powered image generation
- Demonstrated how text prompts can produce Netflix-style campaign designs
- Showcased the transformative potential of generative AI in creative design workflows

### Potential Enhancements
- Add image size/resolution options (landscape, portrait, square)
- Support batch generation of multiple design variations
- Integrate image saving and download functionality
- Add a style selector (minimalist, cinematic, vintage, etc.)